# 00 — Clone Project Repo về Drive

Notebook generic của `colab-venv-bootstrap`. Dùng để clone (lần đầu) hoặc pull (cập nhật) project repo vào Drive.

**Workflow**:
1. Mount Google Drive.
2. Điền 4 biến trong cell config (PROJECT_SLUG, BASE_ROOT, GITHUB_REPO_URL, GITHUB_BRANCH).
3. Nếu repo private: nhập token qua `getpass` (KHÔNG lưu vào notebook).
4. Notebook clone về `$BASE_ROOT/$PROJECT_SLUG`.
5. Sau khi xong: mở `colab/01_bootstrap_env.ipynb` trong repo đã clone để cài venv.

**Tại sao 4 biến trong cell thay vì đọc `config.env`?**

Chicken-and-egg: `config.env` nằm trong repo, mà repo chưa clone về Drive. Notebook 01+ trở đi đọc trực tiếp từ `config.env`.

**Token GitHub** (chỉ cần nếu repo private):
- GitHub → Settings → Developer settings → Personal access tokens → Fine-grained tokens.
- Chọn repo + scope `Contents: read`.
- Token nhập runtime qua getpass, không persist trong git config sau clone.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import base64
import subprocess
from getpass import getpass

# ============================================================
# EDIT 4 biến này theo project của bạn
# ============================================================
PROJECT_SLUG     = 'my_project'                                # tên repo/slug
BASE_ROOT        = '/content/drive/MyDrive'                    # Drive root
GITHUB_REPO_URL  = 'https://github.com/USER/REPO.git'          # URL repo (KHÔNG có token)
GITHUB_BRANCH    = 'main'

# Sync mode khi repo đã tồn tại:
#   'safe' = stash local + ff-only merge (giữ commit local nếu có)
#   'hard' = reset --hard origin/<branch> (BỎ thay đổi local — chỉ dùng khi chắc chắn)
SYNC_MODE        = 'safe'

# Token: để 'REPLACE_WITH_GITHUB_TOKEN' → notebook hỏi runtime (qua getpass).
# Để None hoặc empty string → clone không token (chỉ OK với public repo).
GITHUB_TOKEN     = 'REPLACE_WITH_GITHUB_TOKEN'

# ============================================================
# Derived (không cần sửa)
# ============================================================
REPO_DIR = f'{BASE_ROOT}/{PROJECT_SLUG}'

if GITHUB_TOKEN == 'REPLACE_WITH_GITHUB_TOKEN':
    GITHUB_TOKEN = getpass('Nhập GitHub token (Enter để bỏ qua nếu repo public): ').strip()

USE_AUTH = bool(GITHUB_TOKEN)
if USE_AUTH:
    auth_b64 = base64.b64encode(f'x-access-token:{GITHUB_TOKEN}'.encode()).decode()

os.makedirs(BASE_ROOT, exist_ok=True)
print(f'Repo URL  : {GITHUB_REPO_URL}')
print(f'Repo dir  : {REPO_DIR}')
print(f'Branch    : {GITHUB_BRANCH}')
print(f'Sync mode : {SYNC_MODE}')
print(f'Auth      : {"yes (token)" if USE_AUTH else "no (public)"}')

In [ ]:
def run_cmd(cmd, *, check=True, capture=False, redact=False):
    result = subprocess.run(cmd, check=False, text=True, capture_output=capture)
    if check and result.returncode != 0:
        out = result.stdout or '(empty)'
        err = result.stderr or '(empty)'
        if redact:
            raise RuntimeError(f'Git command failed (token redacted). Exit={result.returncode}\nSTDOUT:\n{out}\nSTDERR:\n{err}')
        raise RuntimeError(f'Command failed. Exit={result.returncode}\nSTDOUT:\n{out}\nSTDERR:\n{err}')
    return result

def git_auth(args, *, check=True, capture=False):
    if USE_AUTH:
        cmd = ['git', '-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth_b64}', *args]
        return run_cmd(cmd, check=check, capture=capture, redact=True)
    return run_cmd(['git', *args], check=check, capture=capture, redact=False)

def git_plain(args, *, check=True, capture=False):
    return run_cmd(['git', *args], check=check, capture=capture, redact=False)

def get_current_branch():
    res = git_plain(['-C', REPO_DIR, 'rev-parse', '--abbrev-ref', 'HEAD'], capture=True)
    b = (res.stdout or '').strip()
    if b and b != 'HEAD':
        return b
    res = git_plain(['-C', REPO_DIR, 'symbolic-ref', '--short', 'refs/remotes/origin/HEAD'],
                    check=False, capture=True)
    if res.returncode == 0 and res.stdout:
        return res.stdout.strip().split('/', 1)[1]
    return GITHUB_BRANCH

if not os.path.isdir(f'{REPO_DIR}/.git'):
    print('[INFO] Clone lần đầu...')
    git_auth(['clone', '--branch', GITHUB_BRANCH, '--recurse-submodules',
              GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    branch = get_current_branch()
    print(f'[INFO] Repo đã có. Branch hiện tại: {branch}')

    dirty = git_plain(['-C', REPO_DIR, 'status', '--porcelain'], capture=True).stdout.strip()
    stashed = False
    if dirty and SYNC_MODE == 'safe':
        print('[INFO] Working tree dirty, auto stash trước khi pull...')
        git_plain(['-C', REPO_DIR, 'stash', 'push', '-u', '-m', 'colab-auto-stash'], check=True)
        stashed = True

    if SYNC_MODE == 'hard':
        print('[WARN] hard sync: reset --hard + clean -fd (BỎ thay đổi local)')
        git_auth(['-C', REPO_DIR, 'fetch', 'origin', branch], check=True)
        git_plain(['-C', REPO_DIR, 'reset', '--hard', f'origin/{branch}'], check=True)
        git_plain(['-C', REPO_DIR, 'clean', '-fd'], check=True)
    else:
        git_auth(['-C', REPO_DIR, 'fetch', 'origin', branch], check=True)
        ff = git_plain(['-C', REPO_DIR, 'merge', '--ff-only', f'origin/{branch}'],
                       check=False, capture=True)
        if ff.returncode != 0:
            raise RuntimeError(
                'Không thể ff-only pull. Có thể branch diverged.\n'
                f"STDERR:\n{ff.stderr or '(empty)'}\n"
                "Đổi SYNC_MODE = 'hard' và chạy lại nếu chấp nhận bỏ thay đổi local."
            )

    # Update submodules (vd colab/bootstrap)
    print('[INFO] Updating submodules...')
    git_auth(['-C', REPO_DIR, 'submodule', 'update', '--init', '--recursive'], check=True)

    if stashed:
        pop = git_plain(['-C', REPO_DIR, 'stash', 'pop'], check=False, capture=True)
        if pop.returncode != 0:
            print('[WARN] Stash pop có conflict, kiểm tra git status thủ công.')

# Reset remote URL — KHÔNG persist token trong git config
git_plain(['-C', REPO_DIR, 'remote', 'set-url', 'origin', GITHUB_REPO_URL], check=True)

print()
print(f'[DONE] Repo sẵn sàng tại: {REPO_DIR}')
print(f'       Mở notebook tiếp theo: {REPO_DIR}/colab/01_bootstrap_env.ipynb')

In [ ]:
# Cell tuỳ chọn: xem cấu trúc thư mục repo
import subprocess
result = subprocess.run(['ls', '-la', REPO_DIR], capture_output=True, text=True)
print(result.stdout)

# Verify colab/bootstrap submodule đã pull về
import os
submodule_dir = f'{REPO_DIR}/colab/bootstrap'
if os.path.isdir(submodule_dir) and os.listdir(submodule_dir):
    print('[OK] Submodule colab/bootstrap có nội dung — sẵn sàng cho notebook 01')
else:
    print('[WARN] Submodule colab/bootstrap rỗng. Chạy thủ công:')
    print(f"   !git -C {REPO_DIR} submodule update --init --recursive")